# Configuration

Set the project root directory below (the root of the git repository that was cloned).

In [1]:
PROJECT_ROOT_DIR = 'C:/Users/sumit/GitRepos/FloodResearch/bhccpx/'

import os
import sys
os.chdir(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))
sys.path.append(os.path.join(PROJECT_ROOT_DIR, 'bhccpx'))

Place this same directory in `DEFAULT:rootdir` in the `bhccpx/bhc_complex.ini` configuration file.

In [2]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import zipfile
import pickle as pkl

import bhc_datautil
import csv2sys
import sys2bhc
import bhc2out

In [3]:
config = bhc_datautil.read_config()

# Downloading Data

Go to the [FFIC NIC Data Download](https://www.ffiec.gov/npw/FinancialReport/DataDownload) site and download all of the CSV ZIP files. This step cannot be automated, as it may require a CAPTCHA.

Once you have done so, place them in the folder you specified in the `DEFAULT::data` directory. You can either unzip them manually (and place the CSV files in the same directory) or run the following utility.

In [9]:
print(f"Data directory: {config.get('DEFAULT', 'datadir')}")

filenames = [
	'CSV_ATTRIBUTES_ACTIVE.zip',
	'CSV_ATTRIBUTES_BRANCHES.zip',
	'CSV_ATTRIBUTES_CLOSED.zip',
	'CSV_RELATIONSHIPS.zip',
	'CSV_TRANSFORMATIONS.zip',
]
for filename in filenames:
	zip_path = os.path.join(config.get('DEFAULT', 'datadir'), filename)
	if os.path.exists(zip_path):
		with zipfile.ZipFile(zip_path, 'r') as zip_ref:
			zip_ref.extractall(config.get('DEFAULT', 'datadir'))
		print(f"Extracted {zip_path}")

Data directory: C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_ACTIVE.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_BRANCHES.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_ATTRIBUTES_CLOSED.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_RELATIONSHIPS.zip
Extracted C:/Users/sumit/GitRepos/FloodResearch/bhccpx/data\CSV_TRANSFORMATIONS.zip


# Parsing the data

We next parse the raw CSV data into NetworkX objects, which are cached locally.  Parsing is an expensive operation, so it is highly desirable to cache the NetworkX objects as intermediate state.

Before running, be sure to check the following configuration options (note that if you change these, you should rerun the cell where the config is read):
- `DEFAULT:parallelcores`:  Each operation may take some time to complete if done sequentially (around 30 minutes with default settings). If you have a strong CPU with multiple cores, we recommend adjusting this option to take advantage of parallel processing
- `csv2sys:asofdate0` and `csv2sys:asofdate1`: these specify the range of asofdates (year/quarter) that will be processed
- `sys2bhc:asoflist`: similar to above; if the list is empty, it will use the same range as specified in `csv2sys`
- `sys2bhc:bhclist`: a list of RSSDs to process; if it is `None`, then all available RSSDs will be processed
- `csv2sys:clearcache` and `sys2bhc:clearcache`: these are `True` by default, so you should only run these cells once

In [ ]:
csv2sys.main(argv=['csv2sys.py'])

In [ ]:
sys2bhc.main(argv=['sys2bhc.py'])

# Extracting a time series
As an example, we will track **SVB Financial Group** (RSSD=1031449) from its inception in 1983 as **Silicon Valley Bancshares** to its collapse in 2023. 

From the cache, we extract SVB's hierarchy objects each quarter for the 40 years (1983-2023). For each hiearchy, we calculate a variety of complexity statistics. 

In [4]:
SVB_rssd = 1031449
asoflist = bhc_datautil.assemble_asofs(config.get('csv2sys', 'asofdate0'), config.get('csv2sys', 'asofdate1'))
print(f"All quarters from {config.get('csv2sys', 'asofdate0')} to {config.get('csv2sys', 'asofdate1')}")

All quarters from 1983Q4 to 2022Q4


In [5]:
df = bhc2out.make_wachwells_comparison([(SVB_rssd, asofdate) for asofdate in asoflist], config=config)

INFO::root::111 - Creating BHC networks
INFO::root::133 - *** Processing Table2 Complete ***


In [6]:
def get_summary_statistics(df: pd.DataFrame, excluded_columns: list[str]) -> pd.DataFrame:
	cols = [col for col in df.columns if col not in excluded_columns]
	summary = pd.DataFrame(columns=['mean', 'std', 'min', '5%', '10%', '25%', '50%', '75%', '90%', '95%', 'max'], dtype=int)
	for col in cols:
		arr = df[col].values
		metrics = [arr.mean(), arr.std(ddof=1), arr.min()] + list(np.percentile(arr, [5, 10, 25, 50, 75, 90, 95])) + [arr.max()]
		summary.loc[col] = metrics
	for colname in ['min', '5%', '10%', '25%', '50%', '75%', '90%', '95%', 'max']:
		summary[colname] = summary[colname].astype(int)
	return summary

In [7]:
get_summary_statistics(df, excluded_columns=['rssd', 'asofdate'])

,mean,std,min,5%,10%,25%,50%,75%,90%,95%,max
Bas_Vertex_count,39.261146,45.714856,2,3,3,3,23,59,97,146,195
Bas_Edge_count,51.777070,63.927703,1,2,2,2,25,79,138,206,259
Bas_Cycle_rank,13.515924,18.361961,0,0,0,0,3,21,42,61,66
Bas_Num_CComp,1.000000,0.000000,1,1,1,1,1,1,1,1,1
Ent_Qfull_B1,48.299363,62.695997,0,0,0,0,22,74,132,200,254
Ent_Qhetr_B1,19.923567,21.450163,0,0,0,0,14,36,43,54,90
Ent_Qfcon_B1,2.707006,2.473607,0,0,0,0,2,5,6,7,7
Ent_Qhcon_B1,1.719745,1.647976,0,0,0,0,1,3,4,5,5
Ent_edgcn_B1,0.573248,0.690632,0,0,0,0,0,1,2,2,2
Ent_DjHom_B1,5.114650,10.001902,0,0,0,0,0,6,18,35,39
